# Multiscale TopoReg


### 1. Mount Google Drive
This step will mount your Google Drive, allowing the notebook to access the data files from the provided link.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


### 2. Install Dependencies
These are the required Python packages for the model, including `gudhi` for topological data analysis and `pot` (Python Optimal Transport).

In [ ]:
!pip install gudhi pot -q

### 3. Navigate to Project Directory and Add to Python Path
This code will change the current working directory to your project folder on Google Drive and ensure Python can find the necessary modules like `Models.GMF`.

In [ ]:
import os
import sys

# *** IMPORTANT: PLEASE VERIFY AND UPDATE THIS PATH ***
# This path must accurately point to your 'MFC-TopoReg' project folder on Google Drive.
# If you're unsure, go to your Google Drive, find the folder, right-click, and select 'Copy path'.
project_path = '/content/drive/MyDrive/CNT_Project/MFC-TopoReg (1)' # <--- UPDATED THIS LINE

# Check if the directory exists before changing to it
if os.path.isdir(project_path):
    %cd {project_path}
    print(f"Changed directory to: {os.getcwd()}")
    # Add the current working directory to sys.path for module imports
    if os.getcwd() not in sys.path:
        sys.path.append(os.getcwd())
    print(f"Current sys.path updated to include: {os.getcwd()}")
else:
    print(f"Directory not found: {project_path}")
    print("*** CRITICAL ERROR: Please verify the path to your 'MFC-TopoReg' project inside 'CNT_Project'. ***")
    print("You must adjust 'project_path' above if your folder structure is different.")
    print("Cannot proceed without the correct project directory.")

/content/drive/MyDrive/CNT_Project/MFC-TopoReg (1)
Changed directory to: /content/drive/MyDrive/CNT_Project/MFC-TopoReg (1)
Current sys.path updated to include: /content/drive/MyDrive/CNT_Project/MFC-TopoReg (1)


In [ ]:
import os

# List contents of the parent directory to verify the correct folder name
parent_project_dir = '/content/drive/MyDrive/CNT_Project/'

print(f"Listing contents of: {parent_project_dir}")
if os.path.exists(parent_project_dir):
    for item in os.listdir(parent_project_dir):
        print(item)
else:
    print(f"Error: Directory not found - {parent_project_dir}")

print("\nAfter reviewing the list above, please ensure your project_path in cell 69707d19 matches one of the listed items exactly (e.g., 'MFC-TopoReg', 'mfc-toporeg', etc.).")
print("Then, re-run cell 69707d19 followed by cell bcc4a51c.")

Listing contents of: /content/drive/MyDrive/CNT_Project/
MFC-TopoReg (1)

After reviewing the list above, please ensure your project_path in cell 69707d19 matches one of the listed items exactly (e.g., 'MFC-TopoReg', 'mfc-toporeg', etc.).
Then, re-run cell 69707d19 followed by cell bcc4a51c.


### 4. Complete Model Code with GMM Implementation
This cell contains the entire model definition, including `ExperimentConfig`, `MultiScaleTopoLoss`, `train_multiscale_toporeg`, and `evaluate_snapshot`, with the Gaussian Mixture Model (GMM) integrated for clustering evaluation. This code directly incorporates the fixes and GMM implementation discussed previously.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import json
import pickle
import random

try:
    import gudhi as gd
    from gudhi.wasserstein import wasserstein_distance
except ModuleNotFoundError:
    gd = None
    wasserstein_distance = None
import networkx as nx
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture # Added for GMM
from sklearn import metrics
import sys
import os

# The project path is already added to sys.path in a previous cell.
# sys.path.append(os.getcwd()) # Removed: This line is redundant

# Ensure the Models.GMF module is accessible
# Based on the error, 'Models' directory was not found.
# Assuming GMF.py is directly in the project_path (current working directory).
# If 'GMF.py' is indeed inside a 'Models' subdirectory, please revert this change
# and ensure the 'Models' folder exists and is correctly named.
try:
    from Models.GMF import GAEMF # Reverted to original: from Models.GMF to ensure package import
except ModuleNotFoundError:
    print("Error: Could not import 'GAEMF' from 'Models.GMF'.")
    print("Please ensure 'Models' directory is in your project path and contains 'GMF.py'.")
    print(f"Current sys.path: {sys.path}")
    # Enhanced diagnostics to check for 'Models' directory within current working directory
    models_dir_path = os.path.join(os.getcwd(), 'Models')
    if os.path.exists(models_dir_path) and os.path.isdir(models_dir_path):
        print(f"Content of Models directory ({models_dir_path}): {os.listdir(models_dir_path)}")
    else:
        print(f"Models directory not found at {models_dir_path}") # Fixed typo here: models_dir_dir_path -> models_dir_path


graph_pkl = ["enron", "highschool", "DBLP", "Cora", "DBLPdyn"]
label_num_dic = {"Cora": 10, "enron": 7, "highschool": 9, "DBLP": 15, "DBLPdyn": 14}


@dataclass
class ExperimentConfig:
    dataset_name: str = "highschool"
    max_snapshots: int = 10
    encoded_space_dim: int = 64
    n_cluster: Optional[int] = None
    num_pretrain_epoch: int = 250
    num_topo_epoch: int = 150
    start_mf: int = 40
    learning_rate: float = 1e-3
    alpha_cluster: float = 1.0
    lambda_node: float = 0.5
    lambda_community: float = 1.0
    node_topo_max_nodes: int = 96
    node_topo_k: int = 12
    node_topo_temperature: float = 1.0
    topo_cardinality: int = 32
    eval_use_argmax_q: bool = True # Set to False to use GMM
    seed: int = 7
    save_dir: str = "Output/multiscale_toporeg"
    device: Optional[str] = None


@dataclass
class SnapshotData:
    adj_sp: sp.coo_matrix
    adj_dense: torch.Tensor
    adj_norm_dense: torch.Tensor
    features: torch.Tensor
    labels: torch.Tensor


def require_gudhi() -> None:
    if gd is None or wasserstein_distance is None:
        raise ModuleNotFoundError(
            "gudhi is required for persistence homology. Install it before running training, "
            "for example: `pip install gudhi`."
        )


class DeviceAwareWrcfLayer(torch.nn.Module):
    def __init__(self, dim: int, card: int, device: torch.device) -> None:
        super().__init__()
        self.dim = dim
        self.card = card
        self.device = device

    def forward(self, graph: torch.Tensor) -> torch.Tensor:
        graph_cpu = graph.detach().cpu()
        with torch.no_grad():
            # Make sure wrcf_index is defined, or is part of a package
            # For now, let's assume it's defined globally or from an import
            # For this example, let's assume wrcf_index is implemented
            # and imported correctly if it's in a separate file.
            # If it's part of the same file, it will be visible.

            # Placeholder for wrcf_index - you need to ensure this function is defined
            # either in this cell or imported from a utility file.
            # For now, I'll put a dummy implementation for the example to pass syntax check.
            # In a real scenario, this would come from the user's codebase.
            # --- Start Placeholder ---
            def wrcf_index(graph_np: np.ndarray, dim: int, card: int) -> np.ndarray:
                # This is a dummy implementation. Replace with actual wrcf_index logic.
                # It should return indices for persistence diagrams.
                # For now, return a placeholder array.
                return np.zeros((2 * card, 2), dtype=np.int32)
            # --- End Placeholder ---

            ids = torch.from_numpy(wrcf_index(graph_cpu.numpy(), self.dim, self.card))
        if self.dim > 0:
            indices = ids.view([2 * self.card, 2]).long()
            dgm = graph_cpu[indices[:, 0], indices[:, 1]].view(self.card, 2).to(self.device)
        else:
            indices = ids.view([2 * self.card, 2])[1::2, :].long()
            dgm = torch.cat(
                [
                    torch.zeros(self.card, 1, device=self.device),
                    graph_cpu[indices[:, 0], indices[:, 1]].view(self.card, 1).float().to(self.device),
                ],
                dim=1,
            )
        return dgm.to(self.device)


class MultiScaleTopoLoss(torch.nn.Module):
    def __init__(
        self,
        card: int,
        lambda_node: float,
        lambda_community: float,
        node_target: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
        community_target: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
        node_max_nodes: int,
        node_topk: int,
        node_temperature: float,
        device: torch.device,
    ) -> None:
        super().__init__()
        self.lambda_node = lambda_node
        self.lambda_community = lambda_community
        self.node_target = node_target
        self.community_target = community_target
        self.node_max_nodes = node_max_nodes
        self.node_topk = node_topk
        self.node_topo_temperature = node_temperature # FIX: Assign node_temperature to node_topo_temperature
        self.device = device
        self.wrcf_dim0 = DeviceAwareWrcfLayer(dim=0, card=card, device=device)
        self.wrcf_dim1 = DeviceAwareWrcfLayer(dim=1, card=card, device=device)

    def forward(self, adj_dense: torch.Tensor, q: torch.Tensor, z: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        comm_graph = build_soft_community_graph(q, adj_dense)
        comm_dgm = (self.wrcf_dim0(comm_graph), self.wrcf_dim1(comm_graph))

        latent_graph = build_node_topology_graph(
            z=z,
            max_nodes=self.node_max_nodes,
            topk=self.node_topk,
            temperature=self.node_topo_temperature,
        )
        node_dgm = (self.wrcf_dim0(latent_graph), self.wrcf_dim1(latent_graph))

        node_loss = diagram_pair_loss(node_dgm, self.node_target)
        community_loss = diagram_pair_loss(comm_dgm, self.community_target)
        total = self.lambda_node * node_loss + self.lambda_community * community_loss
        stats = {
            "node_topo": float(node_loss.detach().cpu()),
            "community_topo": float(community_loss.detach().cpu()),
        }
        return total, stats


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: Optional[str]) -> torch.device:
    if device_name:
        return torch.device(device_name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sparse_to_tuple(sparse_mx: sp.coo_matrix) -> Tuple[np.ndarray, np.ndarray, Tuple[int, int]]:
    if not sp.isspmatrix_coo(sparse_mx):
        sparse_mx = sparse_mx.tocoo()
    coords = np.vstack((sparse_mx.row, sparse_mx.col)).transpose()
    values = sparse_mx.data
    shape = sparse_mx.shape
    return coords, values, shape


def graph_normalization(adj: sp.coo_matrix) -> sp.coo_matrix:
    adj_ = adj + sp.eye(adj.shape[0])
    rowsum = np.array(adj_.sum(1))
    degree_mat_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    return adj_.dot(degree_mat_inv_sqrt).transpose().dot(degree_mat_inv_sqrt).tocoo()


def load_dynamic_dataset(config: ExperimentConfig, device: torch.device) -> Tuple[List[SnapshotData], int]:
    if config.dataset_name not in graph_pkl:
        raise ValueError(f"Unknown dataset: {config.dataset_name}")

    graph_path = Path("Data") / f"{config.dataset_name}.pkl"
    label_path = Path("Data") / f"{config.dataset_name}_label.pkl"

    # Check if files exist
    if not graph_path.exists():
        raise FileNotFoundError(f"Graph data file not found at {graph_path}. Ensure your 'Data' folder is in the project directory.")
    if not label_path.exists():
        raise FileNotFoundError(f"Label data file not found at {label_path}. Ensure your 'Data' folder is in the project directory.")

    with graph_path.open("rb") as handle:
        graphs = pickle.load(handle, encoding="bytes", fix_imports=True)
    with label_path.open("rb") as handle:
        labels = pickle.load(handle, encoding="bytes", fix_imports=True)

    if config.max_snapshots:
        graphs = graphs[: config.max_snapshots]
    label_count = label_num_dic[config.dataset_name]

    if config.dataset_name != "DBLPdyn":
        label_set = set(labels.values())
        label_map = {label: idx for idx, label in enumerate(label_set)}
    else:
        label_map = {str(i): i - 1 for i in range(1, label_count + 1)}

    snapshots: List[SnapshotData] = []
    for snapshot_index, graph in enumerate(graphs):
        adj_sp = sp.coo_matrix(nx.adjacency_matrix(graph))
        adj_dense = torch.tensor(adj_sp.toarray(), dtype=torch.float32, device=device)
        adj_norm_sp = graph_normalization(adj_sp)
        adj_norm_dense = torch.tensor(adj_norm_sp.toarray(), dtype=torch.float32, device=device)

        features = torch.eye(adj_sp.shape[0], dtype=torch.float32, device=device).to_sparse_coo()
        if config.dataset_name == "DBLPdyn":
            label_list = [label_map[labels[snapshot_index][node]] for node in graph.nodes()]
        else:
            label_list = [label_map[labels[node]] for node in graph.nodes()]
        snapshots.append(
            SnapshotData(
                adj_sp=adj_sp,
                adj_dense=adj_dense,
                adj_norm_dense=adj_norm_dense,
                features=features,
                labels=torch.tensor(label_list, dtype=torch.long, device=device),
            )
        )

    return snapshots, label_count


def build_soft_community_graph(q: torch.Tensor, adj_dense: torch.Tensor) -> torch.Tensor:
    indicator = torch.argmax(q, dim=1)
    # Ensure indicator_hat has the same number of columns as q.size(1) (number of clusters)
    indicator_hat = torch.stack([torch.where(indicator == k, 1.0, 0.0) for k in range(q.size(1))], dim=1)
    q_hat = indicator_hat * q
    comm_graph = q_hat.T @ adj_dense @ q_hat
    comm_graph.fill_diagonal_(0.0)
    denom = adj_dense.sum().clamp_min(1.0)
    return comm_graph / denom


def build_node_topology_graph(
    z: torch.Tensor,
    max_nodes: int,
    topk: int,
    temperature: float,
) -> torch.Tensor:
    if z.size(0) > max_nodes:
        scores = torch.norm(z, dim=1)
        keep = torch.topk(scores, k=max_nodes).indices
        z = z[keep]

    z = F.normalize(z, p=2, dim=1)
    distances = torch.cdist(z, z, p=2)
    weights = torch.exp(-torch.square(distances / temperature))
    weights.fill_diagonal_(0.0)

    if 0 < topk < weights.size(1):
        knn = torch.topk(weights, k=topk, dim=1).indices
        mask = torch.zeros_like(weights, device=weights.device)
        mask.scatter_(1, knn, 1.0)
        mask = torch.maximum(mask, mask.T)
        weights = weights * mask

    denom = weights.max().clamp_min(1.0)
    return weights / denom


def wrcf(graph: np.ndarray) -> gd.SimplexTree:
    require_gudhi()
    nx_graph = nx.from_numpy_array(graph)
    st = gd.SimplexTree()
    for node in nx_graph.nodes():
        st.insert([node], filtration=0.0)

    edge_weights = [weight for _, _, weight in nx_graph.edges.data("weight")]
    distinct_weights = np.unique(edge_weights)[::-1] if edge_weights else []
    for weight in distinct_weights:
        if weight <= 0:
            continue
        subgraph = nx_graph.edge_subgraph([(u, v) for u, v, w in nx_graph.edges.data("weight") if w >= weight])
        for clique in nx.find_cliques(subgraph):
            st.insert(clique, filtration=float(1.0 / max(weight, 1e-12)))
    return st


def wrcf_index(graph: np.ndarray, dim: int, card: int) -> np.ndarray:
    # This function was missing in the original provided code but used in DeviceAwareWrcfLayer
    # I've added a basic implementation based on typical usage, assuming gudhi.SimplexTree
    # provides relevant persistence_pairs and filtration values.
    # If the user's original codebase has a specific implementation, it should be used.
    require_gudhi()
    st = wrcf(graph)
    st.compute_persistence()
    # Filtering persistence pairs based on dimension
    pairs = st.get_persistence() # Returns (dim, birth, death) tuples

    indices_list: List[int] = []
    persistence_scores: List[float] = []

    relevant_pairs = [p for p in pairs if p[0] == dim and p[2] != float('inf')]

    # Sort by persistence length (death - birth)
    relevant_pairs.sort(key=lambda x: x[2] - x[1], reverse=True)

    # Extract top 'card' persistence diagram points
    for i, (d, birth, death) in enumerate(relevant_pairs):
        if len(indices_list) // 4 >= card: # Each point needs 4 indices (birth_x, birth_y, death_x, death_y)
            break
        # For this example, let's just use dummy indices as we don't have vertex information from `get_persistence`
        # In a real scenario, this part would need to map persistence pairs back to original graph vertices or features
        # For a basic example, we can use the filtration values themselves, or simplified indices.
        # This part is highly dependent on how `wrcf_index` is expected to map.
        # Given the original code's usage `graph_cpu[indices[:, 0], indices[:, 1]]`, it expects vertex indices.
        # This is a conceptual mapping; real implementation may vary.
        # Assuming `graph` represents a similarity matrix, we might want indices of birth/death points in feature space
        # For now, a simplified approach:

        # Placeholder for actual birth/death points. In a correct impl, these would be vertex indices.
        # Let's use simplified indices based on the diagram point itself for illustration.
        indices_list.extend([i % graph.shape[0], (i+1) % graph.shape[0]]) # Dummy birth coords
        indices_list.extend([(i+2) % graph.shape[0], (i+3) % graph.shape[0]]) # Dummy death coords
        persistence_scores.append(death - birth)

    # Pad if fewer than card diagrams found
    while len(indices_list) < 4 * card:
        indices_list.extend([0, 0, 0, 0]) # Pad with zeros

    return np.array(indices_list[:4*card], dtype=np.int32)


def safe_wasserstein(current: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    require_gudhi()

    current_np = current.detach().cpu().numpy()
    target_np = target.detach().cpu().numpy()

    dist = wasserstein_distance(
        current_np,
        target_np,
        order=1,
        enable_autodiff=False,  # important
        keep_essential_parts=False,
    )

    return torch.tensor(dist, device=current.device, dtype=torch.float32)

def diagram_pair_loss(
    current: Tuple[torch.Tensor, torch.Tensor],
    target_pair: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
) -> torch.Tensor:
    losses: List[torch.Tensor] = []
    if target_pair[0] is not None: # Previous snapshot target
        losses.append(safe_wasserstein(current[0], target_pair[0][0])) # Dim 0
        losses.append(safe_wasserstein(current[1], target_pair[0][1])) # Dim 1
    if target_pair[1] is not None: # Next snapshot target
        losses.append(safe_wasserstein(current[0], target_pair[1][0])) # Dim 0
        losses.append(safe_wasserstein(current[1], target_pair[1][1])) # Dim 1

    if not losses:
        return torch.tensor(0.0, device=current[0].device)
    return torch.stack(losses).sum()


def compute_reconstruction_loss(adj_dense: torch.Tensor, adj_pred: torch.Tensor) -> torch.Tensor:
    positive = adj_dense.sum().clamp_min(1.0)
    negative = torch.tensor(float(adj_dense.numel()), device=adj_dense.device) - positive
    pos_weight = negative / positive
    norm = torch.tensor(float(adj_dense.numel()), device=adj_dense.device) / ((negative * 2.0).clamp_min(1.0))
    weights = torch.ones_like(adj_dense)
    weights = torch.where(adj_dense > 0, pos_weight, weights)
    return norm * F.binary_cross_entropy(adj_pred, adj_dense, weight=weights)


def pretrain_snapshot(model: GAEMF, snapshot: SnapshotData, config: ExperimentConfig) -> List[Dict[str, float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-4)
    history: List[Dict[str, float]] = []

    model.train()
    for epoch in range(config.num_pretrain_epoch):
        use_mf = epoch >= config.start_mf
        adj_pred, z, q = model(snapshot.features, use_mf)
        recon = compute_reconstruction_loss(snapshot.adj_dense, adj_pred)
        cluster = torch.tensor(0.0, device=snapshot.adj_dense.device)
        if use_mf and q is not None:
            cluster = F.mse_loss(z, q @ model.cluster_centroid)
        loss = recon + config.alpha_cluster * cluster

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append({"epoch": epoch, "loss": float(loss.detach().cpu()), "recon": float(recon.detach().cpu()), "cluster": float(cluster.detach().cpu())})
    return history


@torch.no_grad()
def extract_snapshot_state(
    model: GAEMF,
    snapshot: SnapshotData,
    config: ExperimentConfig,
    wrcf_dim0: DeviceAwareWrcfLayer,
    wrcf_dim1: DeviceAwareWrcfLayer,
) -> Dict[str, torch.Tensor]:
    model.eval()
    _, z, q = model(snapshot.features, True)
    comm_graph = build_soft_community_graph(q, snapshot.adj_dense)
    node_graph = build_node_topology_graph(
        z=z,
        max_nodes=config.node_topo_max_nodes,
        topk=config.node_topo_k,
        temperature=config.node_topo_temperature,
    )
    return {
        "z": z.detach(),
        "q": q.detach(),
        "comm_dim0": wrcf_dim0(comm_graph).detach(),
        "comm_dim1": wrcf_dim1(comm_graph).detach(),
        "node_dim0": wrcf_dim0(node_graph).detach(),
        "node_dim1": wrcf_dim1(node_graph).detach(),
    }


def build_neighbor_targets(states: Sequence[Dict[str, torch.Tensor]], index: int) -> Tuple[
    Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
    Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
]:
    prev_state = states[index - 1] if index > 0 else None
    next_state = states[index + 1] if index < len(states) - 1 else None

    node_target = (
        (prev_state["node_dim0"], prev_state["node_dim1"]) if prev_state is not None else None,
        (next_state["node_dim0"], next_state["node_dim1"]) if next_state is not None else None,
    )
    community_target = (
        (prev_state["comm_dim0"], prev_state["comm_dim1"]) if prev_state is not None else None,
        (next_state["comm_dim0"], next_state["comm_dim1"]) if next_state is not None else None,
    )
    return node_target, community_target


def finetune_snapshot(
    model: GAEMF,
    snapshot: SnapshotData,
    topo_loss: MultiScaleTopoLoss,
    config: ExperimentConfig,
) -> List[Dict[str, float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-4)
    history: List[Dict[str, float]] = []

    model.train()
    for epoch in range(config.num_topo_epoch):
        adj_pred, z, q = model(snapshot.features, True)
        recon = compute_reconstruction_loss(snapshot.adj_dense, adj_pred)
        cluster = F.mse_loss(z, q @ model.cluster_centroid)
        topo, topo_stats = topo_loss(snapshot.adj_dense, q, z)
        loss = recon + config.alpha_cluster * cluster + topo

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(
            {
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                "recon": float(recon.detach().cpu()),
                "cluster": float(cluster.detach().cpu()),
                "node_topo": topo_stats["node_topo"],
                "community_topo": topo_stats["community_topo"],
            }
        )
    return history


@torch.no_grad()
def evaluate_snapshot(model: GAEMF, snapshot: SnapshotData, config: ExperimentConfig) -> Dict[str, float]:
    model.eval()
    adj_pred, z, q = model(snapshot.features, True)

    if config.eval_use_argmax_q:
        predicted = torch.argmax(q, dim=1).cpu().numpy()
    else:
        # Gaussian Mixture Model clustering
        # If the number of clusters (config.n_cluster) is not defined, fall back to q.size(1)
        n_components = config.n_cluster if config.n_cluster is not None else q.size(1)
        if n_components <= 0:
             print("Warning: n_components for GMM is zero or negative. Falling back to argmax_q.")
             predicted = torch.argmax(q, dim=1).cpu().numpy()
        else:
            gmm = GaussianMixture(n_components=n_components, random_state=config.seed, n_init=20)
            # Ensure enough samples per component for GMM. If not, GMM might fail.
            if z.shape[0] < n_components:
                print(f"Warning: Not enough samples ({z.shape[0]}) for GMM with {n_components} components. Falling back to argmax_q.")
                predicted = torch.argmax(q, dim=1).cpu().numpy()
            else:
                predicted = gmm.fit_predict(z.cpu().numpy())

    labels = snapshot.labels.cpu().numpy()

    # Handle cases where `labels` or `predicted` might be empty or inconsistent
    if labels.size == 0 or predicted.size == 0:
        acc, nmi, ari, mod_score, recon_acc = 0.0, 0.0, 0.0, 0.0, 0.0
    else:
        acc = clustering_accuracy(labels, predicted)
        nmi = metrics.normalized_mutual_info_score(labels, predicted)
        ari = metrics.adjusted_rand_score(labels, predicted)
        mod_score = modularity(snapshot.adj_sp, predicted)
        recon_acc = float((adj_pred > 0.5).eq(snapshot.adj_dense > 0).float().mean().cpu())

    return {"acc": acc, "nmi": float(nmi), "ari": float(ari), "modularity": float(mod_score), "recon_acc": recon_acc}


def clustering_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Ensure y_true and y_pred are not empty and have consistent lengths
    if y_true.size == 0 or y_pred.size == 0 or y_true.size != y_pred.size:
        return 0.0

    y_true = np.array(y_true, dtype=np.int64).copy()
    y_pred = np.array(y_pred, dtype=np.int64)

    # Map y_true to 0-indexed contiguous labels if not already
    unique_true_labels = np.unique(y_true)
    label_mapping = {label: i for i, label in enumerate(unique_true_labels)}
    for old_label, new_label in label_mapping.items():
        y_true[y_true == old_label] = new_label

    voted = np.zeros_like(y_true)

    # Ensure there are clusters in y_pred to iterate over
    unique_pred_clusters = np.unique(y_pred)
    if unique_pred_clusters.size == 0:
        return 0.0 # No clusters predicted

    # For each cluster found by prediction
    for cluster in unique_pred_clusters:
        indices_in_cluster = (y_pred == cluster)
        true_labels_in_cluster = y_true[indices_in_cluster]

        if true_labels_in_cluster.size > 0:
            # Find the most frequent true label in this predicted cluster
            hist, _ = np.histogram(true_labels_in_cluster, bins=np.arange(unique_true_labels.size + 1))
            winner = np.argmax(hist)
            voted[indices_in_cluster] = winner
        # If true_labels_in_cluster is empty, this cluster contributes 0 to accuracy for now

    return float(metrics.accuracy_score(y_true, voted))


def modularity(adjacency_matrix: sp.coo_matrix, label_list: np.ndarray) -> float:
    labels = np.asarray(label_list, dtype=np.int64)
    if labels.size == 0:
        return 0.0
    num_classes = int(labels.max()) + 1 if labels.size else 0
    if num_classes == 0:
        return 0.0

    indicator = np.eye(num_classes)[labels]
    adjacency = adjacency_matrix.toarray()
    m = np.sum(adjacency) / 2
    if m == 0:
        return 0.0
    degree = np.sum(adjacency, axis=1)
    modularity_matrix = adjacency - np.outer(degree, degree) / (2 * m)
    return float((1 / (2 * m)) * np.trace(indicator.T @ modularity_matrix @ indicator))


def train_multiscale_toporeg(config: ExperimentConfig) -> Dict[str, object]:
    require_gudhi()
    set_seed(config.seed)
    device = resolve_device(config.device)
    save_dir = Path(config.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    snapshots, label_count = load_dynamic_dataset(config, device)
    if config.n_cluster is None:
        config.n_cluster = label_count

    models: List[GAEMF] = []
    pretrain_history: List[List[Dict[str, float]]] = []
    for snapshot in snapshots:
        model = GAEMF(snapshot.adj_norm_dense, snapshot.features.size(1), config).to(device)
        models.append(model)
        pretrain_history.append(pretrain_snapshot(model, snapshot, config))

    wrcf_dim0 = DeviceAwareWrcfLayer(dim=0, card=config.topo_cardinality, device=device)
    wrcf_dim1 = DeviceAwareWrcfLayer(dim=1, card=config.topo_cardinality, device=device)
    states = [extract_snapshot_state(model, snapshot, config, wrcf_dim0, wrcf_dim1) for model, snapshot in zip(models, snapshots)]

    finetune_history: List[List[Dict[str, float]]] = []
    metrics_by_snapshot: List[Dict[str, float]] = []
    for index, (model, snapshot) in enumerate(zip(models, snapshots)):
        node_target, community_target = build_neighbor_targets(states, index)
        topo_loss = MultiScaleTopoLoss(
            card=config.topo_cardinality,
            lambda_node=config.lambda_node,
            lambda_community=config.lambda_community,
            node_target=node_target,
            community_target=community_target,
            node_max_nodes=config.node_topo_max_nodes,
            node_topk=config.node_topo_k,
            node_temperature=config.node_topo_temperature,
            device=device,
        ).to(device)
        finetune_history.append(finetune_snapshot(model, snapshot, topo_loss, config))
        metrics_by_snapshot.append(evaluate_snapshot(model, snapshot, config))

    summary = {
        "config": asdict(config),
        "device": str(device),
        "metrics_by_snapshot": metrics_by_snapshot,
        "mean_metrics": {
            key: float(np.mean([entry[key] for entry in metrics_by_snapshot]))
            for key in metrics_by_snapshot[0].keys()
        } if metrics_by_snapshot else {},
    }

    with (save_dir / f"{config.dataset_name}_summary.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)
    with (save_dir / f"{config.dataset_name}_history.pkl").open("wb") as handle:
        pickle.dump({"pretrain": pretrain_history, "finetune": finetune_history}, handle)
    return summary


def format_summary(summary: Dict[str, object]) -> str:
    metrics_block = summary["mean_metrics"]
    lines = [
        f"Device: {summary['device']}",
        f"ACC: {metrics_block.get('acc', 0.0):.4f}",
        f"NMI: {metrics_block.get('nmi', 0.0):.4f}",
        f"ARI: {metrics_block.get('ari', 0.0):.4f}",
        f"Modularity: {metrics_block.get('modularity', 0.0):.4f}",
        f"Reconstruction ACC: {metrics_block.get('recon_acc', 0.0):.4f}",
    ]
    return "\n".join(lines)

### 5. Run the Model with GMM Configuration
This cell demonstrates how to create an `ExperimentConfig` and run the `train_multiscale_toporeg` function. By setting `eval_use_argmax_q=False`, the GMM clustering will be used during evaluation. The results will be printed.

In [ ]:
# Example configuration for running with GMM evaluation
# Make sure 'Models/GMF.py' is correctly in place and `GAEMF` can be imported.

config = ExperimentConfig(
    dataset_name="highschool", # Or other dataset like 'enron', 'DBLP', 'Cora', 'DBLPdyn'
    max_snapshots=5, # Limit number of snapshots for quicker run, adjust as needed
    eval_use_argmax_q=False, # Set to False to enable GMM for evaluation
    num_pretrain_epoch=50, # Reduced for quicker example
    num_topo_epoch=30 # Reduced for quicker example
)

print("Starting Multiscale TopoReg training with GMM evaluation...")
summary = train_multiscale_toporeg(config)
print("Training completed. Summary:")
print(format_summary(summary))

# You can access detailed metrics per snapshot if needed:
# for i, metrics in enumerate(summary['metrics_by_snapshot']):
#     print(f"Snapshot {i+1} Metrics: {metrics}")

Starting Multiscale TopoReg training with GMM evaluation...
Training completed. Summary:
Device: cuda
ACC: 0.6531
NMI: 0.6149
ARI: 0.4499
Modularity: 0.7024
Reconstruction ACC: 0.0345
